<a href="https://colab.research.google.com/github/angeliaanne/DS2002FA26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO:

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4, f'expected 4 rows, got {len(lines)}'
assert lines['qty'].sum() == 7, f'expected qty to sum to 7, got {lines["qty"].sum()}'
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendors = pd.DataFrame(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']
print('unmatched vendor(s):', unmatched['vendor_id'].unique())
print('units attached to the unmatched vendor:', unmatched['qty'].sum())
joined

unmatched vendor(s): []
units attached to the unmatched vendor: 0


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
# There's no unmatched vendor in this dataset (see Q2)so nothing tofill in.
#If there had been one, I would NOT fill its name with 0 units/drop it silently.
# I'd fillna the name with something like 'Unknown (<vendor_id>)' so its units still show up in the total instead ofvanishing, the same way we handled the missing-revenue vendor in the studio.
by_vendor = joined.groupby('name')['qty'].sum().sort_values(ascending=False)
by_vendor

,qty
name,
Hoos Burgers,6
Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
by_zone = joined.groupby('zone')['qty'].sum().sort_values(ascending=False)
print(by_zone)
print('zone total:', by_zone.sum())

zone
A    7
Name: qty, dtype: int64
zone total: 7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
# json_normalize crashes if even one record is missing the record_path key entirely, so give every record a 'lines' key first -order 5 gets an empty list, which just means it contributes 0 rows (it had no items to flatten)
safe = [{**rec, 'lines': rec.get('lines', [])} for rec in ragged_json]

ragged_lines = pd.json_normalize(safe, record_path='lines', meta=['order', 'vendor_id'])

# Don't fillna(0) here -- a missing qty (order 6, Fries) isn't the same thingas an order for 0 Fries.
#Leaving it as NaN keeps "we dk how many" distinguishable from "they ordered zero," which is what Q6/write-up (b) asks about
ragged_lines

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** When we organize sales data, it makes a lot of sense to have one row for each item sold. This way, I can easily see how many of each item we sold. If we used one row per order instead, like if someone ordered two cheeseburgers and a soda all together, it would just show up as one entry with a list of items. Every time I want to find the total for a specific item, I’d have to sift through that list, which can get cumbersome. By flattening the data once at the start, I can later group it by item, vendor, or zone with a simple command instead of going through layers of lists.


**b)** A missing quantity means we don't actually know how many were ordered, maybe the data just didn't get recorded. On the other hand, if I mark it as zero, that means we definitely know no one ordered that item. If I treat the missing quantities as zero, it can make my totals look lower than they are, masking real orders. Plus, if I try to calculate an average quantity per order, those missing numbers would make it seem like there were fewer sales overall, skewing the results in a misleading way.